# Lesson 24 — OTA：校验、候选槽与失败恢复

`%ota_sandbox` 真实创建 firmware 包、计算 SHA-256、写入非活动槽，并在健康检查失败时回滚。它不写 ESP32 Flash，也不替代签名、secure boot、分区表或 bootloader。

**学习顺序：** 先运行 guided，再修改，故意运行 broken，最后自己完成 exercise。答案 Notebook 独立保存。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

## 1. Guided

先观察面板输出，再读 C++。

In [ ]:
%%academy_lab 24-safe-ota guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){if(device_.input.fault){device_.output.error=true;device_.output.state="rolled-back";return;}if(device_.input.text.empty()){device_.output.error=true;device_.output.state="no-package";return;}device_.output.state="verify-before-activate";device_.output.display="SHA-256 -> inactive slot -> health check -> confirm";} private:Device& device_;}; }

## 2. Modify

先运行 `%ota_sandbox` 的“生成并校验包”，再点击“激活候选槽”和“失败启动”，观察原槽保留。

In [ ]:
%%academy_lab 24-safe-ota modify
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){if(device_.input.fault){device_.output.error=true;device_.output.state="rolled-back";return;}if(device_.input.text.empty()){device_.output.error=true;device_.output.state="no-package";return;}device_.output.state="verify-before-activate";device_.output.display="SHA-256 -> inactive slot -> health check -> confirm";} private:Device& device_;}; }

## 3. Broken

下面的检查应该失败；先观察故障证据，再查看分层提示。

In [ ]:
%%academy_lab 24-safe-ota broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){device_.output.state=device_.input.text.empty()?"no-package":"activated-without-check";} private:Device& device_;}; }

## 4. Build

要求：先验证包内容，再激活 inactive slot；健康检查失败回滚，成功才确认。不得把文件名或 UI 状态当作安全证明。

提示 1：manifest 的 digest 必须等于实际 bytes 的 SHA-256。提示 2：候选槽与 current 槽分离。提示 3：pending boot 只有健康后确认。

In [ ]:
%%academy_lab 24-safe-ota exercise exercise
#pragma once
#include "academy/device.hpp"
namespace academy { class Controller { public: explicit Controller(Device& d):device_(d){} void tick(){ /* TODO: 你的实现 */ } private: Device& device_; }; }


## 5. 真实 OTA 沙箱

In [ ]:
%ota_sandbox

## 6. 反思

写下：这个模型在 host 上证明了什么？哪些行为必须买板后重新验证？